In [1]:
from dotenv import load_dotenv
import getpass
import os

if not os.getenv("HUGGINGFACEHUB_API_TOKEN"):
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = getpass.getpass("Enter your token:")

load_dotenv()

True

In [2]:
import importlib
import eval_utils

importlib.reload(eval_utils)

/home/a.zhang@PTW.Maschinenbau.TU-Darmstadt.de/project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<module 'eval_utils' from '/home/a.zhang@PTW.Maschinenbau.TU-Darmstadt.de/project/RAGAS/eval_utils.py'>

In [3]:
from eval_utils import load_llm, load_embeddings, load_docs

# load chat model
chat_model = load_llm(model_path="./hf_models/Llama-3.1-8B-Instruct")

# load embedding model
embedding = load_embeddings(model_id="sentence-transformers/all-MiniLM-L6-v2",
                            cache_dir="hf_models/embeddings/all-MiniLM-L6-v2")

# load docs
docs = load_docs("data/mk4s_manuel")

Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.36it/s]
Device set to use cuda:0
/home/a.zhang@PTW.Maschinenbau.TU-Darmstadt.de/project/RAGAS/eval_utils.py:55: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceEmbeddings(
100%|██████████| 8/8 [00:00<00:00, 9947.95it/s]


In [4]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=256, 
                                               chunk_overlap=32,
                                               separators=["\n## ","\n### ","\n#### ", "\n\n", ".", "\n", " "],
                                               is_separator_regex=False,
                                               keep_separator=False,)
chunks = text_splitter.split_documents(docs)

Initialize Hybrid Retriever

In [5]:
import time
from typing import List, Set
import pandas as pd
from langchain_community.vectorstores import FAISS
from langchain_core.retrievers import BaseRetriever
from langchain.retrievers.ensemble import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_core.prompts import PromptTemplate
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_core.runnables import RunnableLambda
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from langchain_core.documents import Document

# set hybrid retriever as base retriever
vector_storage = FAISS.from_documents(chunks, embedding)

bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 10

faiss_retriever = vector_storage.as_retriever(
                                search_type="mmr",
                                search_kwargs={"k": 10, "lambda_mult": 0.7}
                                )

class HybridRetriever(BaseRetriever):
    
    base_retriever: BaseRetriever
    final_k: int = 10

    def _get_relevant_documents(
        self,
        query: str,
        *,
        run_manager: CallbackManagerForRetrieverRun,
    ) -> List[Document]:
        
        docs = self.base_retriever.get_relevant_documents(query)

        # deduplication
        seen: Set[str] = set()
        unique_docs: List[Document] = []
        for d in docs:
            if d.page_content not in seen:
                seen.add(d.page_content)
                unique_docs.append(d)

        return unique_docs[:self.final_k]
    
ensemble_retriever = EnsembleRetriever(
        retrievers=[bm25_retriever, faiss_retriever], weights=[0.4, 0.6]
    )

hybrid_retriever = HybridRetriever(
                    base_retriever=ensemble_retriever,
                    final_k=10,)

Multi-Query Retrieval

In [6]:
mq_prompt = PromptTemplate(
    input_variables=["question"],
    template=(
        """ You are a helpful research assistant. 
        Given the user's question below, generate **exactly three** alternative search queries. 

        Each query should capture a different aspect or phrasing of the same information need.
        Avoid redundancy, keep them short and clear.
        User question: {question}
        Provide the 3 reformulated search queries, each on a new line:
        """
    ),
)

# Create MultiQueryRetriever，prompt and LLM
mq_retriever_raw = MultiQueryRetriever.from_llm(
    retriever=hybrid_retriever,
    llm=chat_model,
    prompt=mq_prompt,
    include_original=True,
)

class DedupRetriever(BaseRetriever):

    base_retriever: BaseRetriever

    def _get_relevant_documents(
        self,
        query: str,
        *,
        run_manager: CallbackManagerForRetrieverRun,
    ) -> List[Document]:

        docs = self.base_retriever.get_relevant_documents(query)

        # deduplication
        seen: Set[str] = set()
        unique_docs: List[Document] = []

        for d in docs:
            if d.page_content not in seen:
                seen.add(d.page_content)
                unique_docs.append(d)

        return unique_docs
    
mq_retriever = DedupRetriever(base_retriever=mq_retriever_raw)

In [ ]:
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough,RunnableLambda
from langchain.schema.output_parser import StrOutputParser

template = """You are a helpful assistant specializing in 3D printing operations.  
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know. 
Use two sentences maximum and keep the answer concise.
Question: {question} 
Context: {context} 
Answer:
"""
def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

prompt = ChatPromptTemplate.from_template(template)

mq_rag_chain = (
    {"context": mq_retriever | RunnableLambda(format_docs),
     "question": RunnablePassthrough()}
    | prompt
    | chat_model
    | StrOutputParser()
)


In [ ]:
import pandas as pd
import time
from ragas import EvaluationDataset

def get_eval_ds(path, retriever, rag_chain):

    testset = pd.read_csv(path)
    queries = testset["user_input"].to_list()
    references = testset["reference"].to_list()

    # generate evaluation dataset
    rs_times = []
    dataset = []

    for query, reference in zip(queries, references):

        
        relevant_docs = [doc.page_content for doc in retriever.invoke(query)]
        # measure response time
        t0 = time.perf_counter()
        response = rag_chain.invoke(query)
        t1 = time.perf_counter()
        
        dataset.append(
            {
                "user_input": query,
                "retrieved_contexts": relevant_docs,
                "response": response,
                "reference": reference,
            }
        )
        rs_times.append(t1 -t0)

    eval_ds = EvaluationDataset.from_list(dataset)

    return eval_ds, rs_times

In [18]:
from ragas import evaluate, RunConfig
from ragas.metrics import LLMContextPrecisionWithReference, LLMContextRecall, AnswerAccuracy, AnswerRelevancy, Faithfulness, FactualCorrectness
from eval_utils import load_eval_model

def get_eval_result(eval_ds, metrics):
    evaluator_llm = load_eval_model()

    run_config = RunConfig(
            timeout=600,      
            max_workers=2,    
            max_retries=10,    
            max_wait=600,     
        )
    result = evaluate(
            dataset=eval_ds,
            metrics=metrics,
            llm=evaluator_llm,
            batch_size=1,
            allow_nest_asyncio=False, 
            run_config=run_config,
        )
    
    return result

In [14]:
# select metrics
metrics=[LLMContextPrecisionWithReference(),
         LLMContextRecall(), 
         AnswerAccuracy(),
         AnswerRelevancy(),
         Faithfulness(),]

In [ ]:
data_path = "evaluate_dataset/test_dataset.csv"

eval_ds, response_time = get_eval_ds(data_path, mq_retriever, mq_rag_chain)
result = get_eval_result(eval_ds, metrics)
print(f"multi query:{result}")

In [ ]:
df = result.to_pandas()
df["response_time"] = response_time
df.to_csv(f"./evaluate_results/05_query_test/multi_query.csv")

HyDE

In [10]:
from typing import Any

class HyDERetriever(BaseRetriever):
    llm: Any
    retriever: BaseRetriever

    def _get_relevant_documents(
        self, 
        query: str,
        *,
        run_manager: CallbackManagerForRetrieverRun,
    ) -> List[Document]:

        # create hypothetical documentation
        hyde_prompt = f"Given the following question, generate a hypothetical answer(1–2 sentences) that could plausibly appear in a technical manual.:\n\n{query}\n\nHypothetical answer:"
        hypo_doc = self.llm.invoke(hyde_prompt)

        hypothetical_text = (
            hypo_doc.content 
            if hasattr(hypo_doc, "content") 
            else str(hypo_doc)
        )

        # Use hypothetical documentation to replace query
        docs = self.retriever.get_relevant_documents(hypothetical_text)

        return docs

In [11]:
hyde_retriever = HyDERetriever(
    llm=chat_model,      
    retriever=hybrid_retriever 
)

In [ ]:
hyde_rag_chain = (
    {"context": hyde_retriever | RunnableLambda(format_docs), 
     "question": RunnablePassthrough()}
    | prompt
    | chat_model
    | StrOutputParser()
)

In [30]:
data_path = "evaluate_dataset/test_dataset.csv"

eval_ds, response_time = get_eval_ds(data_path, hyde_retriever, hyde_rag_chain)
result = get_eval_result(eval_ds, metrics)
print(f"hyde:{result}")

Evaluating: 100%|██████████| 200/200 [18:53<00:00,  5.67s/it]


hyde:{'llm_context_precision_with_reference': 0.5607, 'context_recall': 0.6383, 'nv_accuracy': 0.3875, 'answer_relevancy': 0.9174, 'faithfulness': 0.6190}


In [ ]:
df = result.to_pandas()
df["response_time"] = response_time
df.to_csv(f"./evaluate_results/05_query_test/hyde.csv")

Step Back Prompting

In [20]:
from langchain.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain.schema.output_parser import StrOutputParser

# few-shot examples
examples = [
    {
        "input": "How do I preheat the nozzle on FDM printer models?",
        "output": "How do users typically adjust temperature settings on an FDM printer?"
    },
    {
        "input": "what can i do to stop the first layer from detaching?",
        "output": "What general factors affect first-layer adhesion in 3D printing?"
    },
    {
        "input": "How does filament moisture affect stringing and print quality?",
        "output": "What general material-related factors influence the quality of 3D printed parts?"
    },
    {
        "input": "How can I improve print quality for objects with overhangs?",
        "output": "What general factors influence the quality of prints with challenging geometries?"
    }
]


# Example Prompt
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

# Step-back prompt template
step_back_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an expert at simplifying questions into more generic step-back questions that improve document retrieval. Here are examples:"
        ),
        few_shot_prompt,
        ("human", "{question}")
    ]
)

# Runnable generator of step-back question
step_back_question_gen = step_back_prompt | chat_model | StrOutputParser()


In [21]:
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from typing import List, Any

class StepBackRetriever(BaseRetriever):
    base_retriever: BaseRetriever
    step_back_gen: Any 

    # --- normal retrieval (for RAGAS) ---
    def _get_relevant_documents(
        self,
        query: str,
        *,
        run_manager: CallbackManagerForRetrieverRun,
    ) -> List[Document]:
        # normal retrieval (used by RAGAS + RAG chain)
        return self.base_retriever.get_relevant_documents(query)

    # --- additional method for step-back retrieval ---
    def get_step_back_context(self, query: str) -> List[Document]:
        step_q = self.step_back_gen.invoke({"question": query})
        return self.base_retriever.get_relevant_documents(step_q)


In [22]:
stepback_retriever = StepBackRetriever(
    base_retriever=hybrid_retriever,   
    step_back_gen=step_back_question_gen,
)

In [ ]:
stepback_rag_chain = (
    {"context": stepback_retriever | RunnableLambda(format_docs),
     "question": RunnablePassthrough()}
    | prompt
    | chat_model
    | StrOutputParser()
)

In [24]:
data_path = "evaluate_dataset/test_dataset.csv"

eval_ds, response_time = get_eval_ds(data_path, stepback_retriever, stepback_rag_chain)
result = get_eval_result(eval_ds, metrics)
print(f"step back:{result}")

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Evaluating: 100%|██████████| 200/200 [17:54<00:00,  5.37s/it]


step back:{'llm_context_precision_with_reference': 0.6238, 'context_recall': 0.5532, 'nv_accuracy': 0.4313, 'answer_relevancy': 0.9449, 'faithfulness': 0.7882}


In [ ]:
df = result.to_pandas()
df["response_time"] = response_time
df.to_csv(f"./evaluate_results/05_query_test/step_back.csv")

Technical Disambiguation

In [27]:
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser

examples = [
    {
        "input": "What is extruder blob?",
        "output": "What is an extruder blob forming around the hotend of an FDM printer?"
    },
    {
        "input": "My first layer keeps detaching, what can I do?",
        "output": "How can I improve first-layer bed adhesion on an FDM 3D printer?"
    },
    {
        "input": "How do I fix stringing?",
        "output": "How do I fix filament stringing caused by insufficient retraction in FDM printing?"
    }
]

example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}")
    ]
)

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

disamb_prompt = ChatPromptTemplate.from_messages(
    [
        ("system",
         "You are an expert in FDM 3D printing terminology. Rewrite the user's question by making its technical intent explicit without expanding its meaning."),
        few_shot_prompt,
        ("human", "{query}")
    ]
)

technical_disambiguator = disamb_prompt | chat_model | StrOutputParser()

In [28]:
class TechnicalDisambiguationRetriever(BaseRetriever):
    base_retriever: BaseRetriever
    disambiguator: Any  

    def _get_relevant_documents(
        self,
        query: str,
        *,
        run_manager: CallbackManagerForRetrieverRun,
    ) -> List[Document]:
        return self.base_retriever.get_relevant_documents(query)

    def get_disambiguated_context(self, query: str) -> List[Document]:
        rewritten = self.disambiguator.invoke({"query": query})
        return self.base_retriever.get_relevant_documents(rewritten)

In [29]:
disamb_retriever = TechnicalDisambiguationRetriever(
    base_retriever=hybrid_retriever,              
    disambiguator=technical_disambiguator 
)

In [ ]:
from langchain.schema.runnable import RunnablePassthrough, RunnableLambda
disamb_rag_chain = (
    {
        "context": disamb_retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough(),
    }
    | prompt 
    | chat_model
    | StrOutputParser()
)


In [32]:
data_path = "evaluate_dataset/test_dataset.csv"

eval_ds, response_time = get_eval_ds(data_path, disamb_retriever, disamb_rag_chain)
result = get_eval_result(eval_ds, metrics)
print(f"Disambiguation:{result}")

Evaluating: 100%|██████████| 200/200 [16:23<00:00,  4.92s/it]


Disambiguation:{'llm_context_precision_with_reference': 0.6254, 'context_recall': 0.5839, 'nv_accuracy': 0.4313, 'answer_relevancy': 0.9495, 'faithfulness': 0.7674}


In [ ]:
df = result.to_pandas()
df["response_time"] = response_time
df.to_csv(f"./evaluate_results/05_query_test/disambiguation.csv")

Query Decomposition

In [12]:
from typing import List, Optional
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.language_models import BaseLanguageModel


DECOMP_PROMPT = """
You are an expert at converting user questions into subqueries.

Your task is query decomposition.

Given a user question:
1. Decide whether it requires single-hop retrieval (one direct lookup) or multi-hop retrieval (multiple distinct lookups).
2. If multi-hop, break the question into clear, self-contained sub-questions needed to answer it.
3. If single-hop, return only the original question as the sole sub-question.

Do not over-decompose. Prefer single-hop when uncertain.
Do not rephrase unfamiliar acronyms or invented words.

Return the list of sub-questions only, one per line, without explanations.

User question:
{question}
"""


class QueryDecompositionRetriever(BaseRetriever):
    
    base_retriever: BaseRetriever
    llm: BaseLanguageModel
    prompt: ChatPromptTemplate = ChatPromptTemplate.from_template(DECOMP_PROMPT)
    debug: bool = False

    def _parse_subqueries(self, text: str) -> List[str]:
        """
        Output is expected to be plain text, multiple lines.
        Example:
            line 1
            line 2
        """
        lines = [x.strip() for x in text.split("\n") if x.strip()]
        return lines if lines else []

    def _get_relevant_documents(
        self,
        query: str,
        *,
        run_manager=None,
    ) -> List[Document]:

        #  Ask LLM to decompose
        messages = self.prompt.format_messages(question=query)
        result = self.llm.invoke(messages)
        text = result.content if hasattr(result, "content") else str(result)

        subqueries = self._parse_subqueries(text)

        # Fallback: if parsing fails, treat as single-hop
        if not subqueries:
            subqueries = [query]

        # For each sub-query, call base retriever
        docs = []
        for sq in subqueries:
            retrieved = self.base_retriever.get_relevant_documents(sq)
            for d in retrieved:
                # annotate which subquery retrieved it
                d = Document(page_content=d.page_content, metadata={**d.metadata, "subquery": sq})
                docs.append(d)
        if self.debug:
            print("\n=== Query Decomposition Debug ===")
            print("Original query:", query)
            print("Subqueries:")
            for i, sq in enumerate(subqueries, 1):
                print(f"  {i}. {sq}")
            print("=================================\n")

        return docs
    
    

In [13]:
qd_retriever = QueryDecompositionRetriever(
    base_retriever=hybrid_retriever,
    llm=chat_model,
    debug=False
)

In [14]:
from langchain_community.document_transformers import EmbeddingsRedundantFilter
from langchain.retrievers.document_compressors import DocumentCompressorPipeline
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever

redundant_filter = EmbeddingsRedundantFilter(embeddings=embedding, similarity_threshold=0.9)
pipeline_compressor = DocumentCompressorPipeline(
        transformers=[redundant_filter]
    )

compression_retriever = ContextualCompressionRetriever( 
        base_compressor=pipeline_compressor,
        base_retriever=qd_retriever
    )


In [ ]:
qd_rag_chain = (
        {
            "context": qd_retriever | RunnableLambda(format_docs),
            "question": RunnablePassthrough()
        }
        | prompt
        | chat_model
        | StrOutputParser()
    )
    

In [16]:
qd_rag_chain.invoke("What are the recommended steps for checking belt tension and ensuring proper motor pulley alignment on both the Original Prusa MK-series and Original Prusa XL printers?")

'To check belt tension on the Original Prusa MK-series and Original Prusa XL printers, use the belt tuner or check the belt tension by hand, ensuring the belt clamps are in good visible status and there are no obstructions in the path of the extruder or heatbed. For proper motor pulley alignment, make sure the X and Y motors are tightened in the motor mount, the pulley is secured on the motor shaft and aligned with the pulley on the opposite end, and the pulley can move freely.'

In [ ]:
data_path = "evaluate_dataset/test_dataset.csv"

eval_ds, response_time = get_eval_ds(data_path,qd_retriever, qd_rag_chain)

In [ ]:
result = get_eval_result(eval_ds, metrics)
print(f"QueryDecomposition:{result}")

In [ ]:
df = result.to_pandas()
df["response_time"] = response_time
df.to_csv(f"./evaluate_results/05_query_test/decomposition.csv")